
# 05. Expert system + explanation / conflict layer

## Цель

В `04` мы зафиксировали лучшую OOF-архитектуру:

- calibrated L1;
- calibrated CatBoost;
- weighted ensemble;
- clinical rules не заставляем участвовать в prediction, если meta-training не даёт им вес.

В этом ноутбуке **не переобучаем ML-модели**.

Задача — построить поверх уже зафиксированных ensemble predictions первый слой экспертной системы:

1. `evidence`;
2. `conflicts`;
3. `rule_strength`;
4. `confidence`;
5. `recommended_next_tests`;
6. единый JSON-like `explanation_payload`.

Это будет прототип того payload, который позже сможет возвращать API.



## 1. Архитектура этого этапа

```text
calibrated L1 ─┐
               ├─> weighted ensemble ─> prediction/probability
calibrated CB ─┘                         │
                                        ├─> clinical evidence
clinical rules ──────────────────────────┤
                                        ├─> conflicts
                                        ├─> confidence
                                        └─> next-test suggestions
```

### Важное ограничение

Clinical rules здесь **не меняют уже полученный диагноз**.

Они работают как:

- evidence layer;
- conflict/safety layer;
- explanation layer.

Если позже внешний validation покажет, что rules помогают устойчивости, архитектуру можно пересмотреть.


In [1]:

from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)


## 2. Пути проекта и загрузка зафиксированных OOF predictions

In [2]:

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p.resolve()
    return None

DATA_PATH = first_existing([
    "../data/raw/deficiency_anemia.csv",
    "data/raw/deficiency_anemia.csv",
    "../../data/raw/deficiency_anemia.csv",
])

ENSEMBLE_OOF_PATH = first_existing([
    "../artifacts/ensemble/ensemble_oof_predictions.csv",
    "artifacts/ensemble/ensemble_oof_predictions.csv",
    "../../artifacts/ensemble/ensemble_oof_predictions.csv",
])

if DATA_PATH is None:
    raise FileNotFoundError("Не найден data/raw/deficiency_anemia.csv")

if ENSEMBLE_OOF_PATH is None:
    raise FileNotFoundError(
        "Не найден ensemble_oof_predictions.csv. "
        "Сначала запусти 04_rules_calibration_ensemble_frozen.ipynb полностью."
    )

df = pd.read_csv(DATA_PATH)
oof = pd.read_csv(ENSEMBLE_OOF_PATH)

print("Dataset:", DATA_PATH, df.shape)
print("OOF:", ENSEMBLE_OOF_PATH, oof.shape)


Dataset: /home/sinopah/meditron/data/raw/deficiency_anemia.csv (840, 48)
OOF: /home/sinopah/meditron/artifacts/ensemble/ensemble_oof_predictions.csv (840, 30)



## 3. Подключаем clinical rule module

Файл `clinical_rules.py` положи в:

```text
meditron/
└── ml/
    └── clinical_rules.py
```

Ноутбук ниже автоматически пытается найти модуль и в `../ml`, и в `ml`.


In [3]:

module_dir = first_existing([
    "../ml",
    "ml",
    "../../ml",
])

if module_dir is None:
    raise FileNotFoundError(
        "Не найдена папка ml. Создай ml/clinical_rules.py "
        "из приложенного файла clinical_rules.py."
    )

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from clinical_rules import (
    CLINICAL,
    RULE_TARGETS,
    anemia_rule,
    compute_rule_scores,
    build_evidence,
    build_conflicts,
    recommended_next_tests,
    confidence_label,
)

print("Clinical rules imported successfully.")


Clinical rules imported successfully.


## 4. Проверяем согласованность пациентов

In [4]:

assert df["patient_id"].is_unique
assert oof["patient_id"].is_unique

work = df.merge(
    oof,
    on="patient_id",
    how="inner",
    validate="one_to_one",
)

assert len(work) == len(df) == len(oof), (
    len(df),
    len(oof),
    len(work),
)

print("Patients matched:", len(work))


Patients matched: 840



## 5. Формируем explanation payload для одного пациента

Payload состоит из:

```text
patient_id
prediction
confidence
deficiencies
evidence
conflicts
recommended_next_tests
```

`confidence` — это не вероятность «истинности диагноза».
Это удобная продуктовая категория на основе максимальной ensemble probability.


In [5]:

def patient_probabilities(row):
    return {
        target: float(row[f"{target}__probability"])
        for target in RULE_TARGETS
    }


def patient_predictions(row):
    return {
        target: int(row[f"{target}__prediction"])
        for target in RULE_TARGETS
    }


def patient_rule_scores(row):
    # Пересчитываем из исходных анализов через отдельный clinical_rules.py.
    # Это позволяет проверить, что rule config действительно вынесен из notebook 04.
    return compute_rule_scores(row)


def build_explanation_payload(row):
    probabilities = patient_probabilities(row)
    predictions = patient_predictions(row)
    rule_scores = patient_rule_scores(row)

    top_target = max(probabilities, key=probabilities.get)
    top_probability = probabilities[top_target]

    deficiencies = {
        target: {
            "probability": round(probabilities[target], 4),
            "prediction": bool(predictions[target]),
            "rule_score": round(rule_scores[target], 4),
        }
        for target in RULE_TARGETS
    }

    payload = {
        "patient_id": row["patient_id"],
        "prediction": {
            "anemia_class": row["ensemble_class_pred"],
            "deficiency_cause": row["ensemble_cause_pred"],
            "anemia": bool(row["anemia_rule"]),
        },
        "confidence": {
            "top_binary_target": top_target,
            "top_probability": round(top_probability, 4),
            "level": confidence_label(top_probability),
        },
        "deficiencies": deficiencies,
        "evidence": build_evidence(row),
        "conflicts": build_conflicts(
            probabilities=probabilities,
            predictions=predictions,
            rule_scores=rule_scores,
        ),
        "recommended_next_tests": recommended_next_tests(
            row,
            probabilities,
        ),
        "disclaimer": (
            "Decision-support output. "
            "Не является самостоятельным клиническим диагнозом и требует интерпретации врачом."
        ),
    }

    return payload


## 6. Пример payload

In [6]:

example_payload = build_explanation_payload(work.iloc[0])

print(json.dumps(
    example_payload,
    ensure_ascii=False,
    indent=2,
))


{
  "patient_id": "P000001",
  "prediction": {
    "anemia_class": "B12_deficiency_no_anemia",
    "deficiency_cause": "B12_deficiency",
    "anemia": false
  },
  "confidence": {
    "top_binary_target": "B12_deficiency",
    "top_probability": 0.8282,
    "level": "moderate"
  },
  "deficiencies": {
    "iron_deficiency": {
      "probability": 0.0009,
      "prediction": false,
      "rule_score": 0.05
    },
    "B12_deficiency": {
      "probability": 0.8282,
      "prediction": true,
      "rule_score": 0.15
    },
    "folate_deficiency": {
      "probability": 0.1302,
      "prediction": false,
      "rule_score": 0.95
    },
    "B6_deficiency": {
      "probability": 0.0033,
      "prediction": false,
      "rule_score": 0.5
    },
    "copper_deficiency": {
      "probability": 0.0007,
      "prediction": false,
      "rule_score": 0.5
    },
    "inflammation_anemia": {
      "probability": 0.0002,
      "prediction": false,
      "rule_score": 0.05
    }
  },
  "evidence":


## 7. Строим payload для всех пациентов

Это нужно не для обучения, а для проверки:

- нет ли слишком большого количества conflicts;
- где чаще возникают конфликты;
- какие анализы чаще всего предлагаются как недостающие;
- не выдаёт ли explanation layer нелогичные сообщения.


In [7]:

payloads = [
    build_explanation_payload(row)
    for _, row in work.iterrows()
]

print("Payloads:", len(payloads))


Payloads: 840


## 8. Анализ conflicts

In [8]:

conflict_rows = []

for payload in payloads:
    for conflict in payload["conflicts"]:
        conflict_rows.append({
            "patient_id": payload["patient_id"],
            **conflict,
        })

conflict_df = pd.DataFrame(conflict_rows)

if len(conflict_df) == 0:
    print("Сильных конфликтов по текущим product-thresholds нет.")
else:
    display(
        conflict_df
        .groupby(["target", "type"])
        .size()
        .rename("n")
        .reset_index()
        .sort_values("n", ascending=False)
    )

    display(conflict_df.head(20))


,target,type,n
2,inflammation_anemia,ml_negative_rule_positive,126
0,B12_deficiency,ml_positive_rule_negative,101
4,iron_deficiency,ml_positive_rule_negative,22
3,iron_deficiency,ml_negative_rule_positive,12
1,folate_deficiency,ml_negative_rule_positive,7


,patient_id,target,type,severity,ensemble_probability,rule_score,message
0,P000001,B12_deficiency,ml_positive_rule_negative,review,0.828236,0.15,Высокая ML-вероятность при слабой поддержке clinical rule layer.
1,P000001,folate_deficiency,ml_negative_rule_positive,review,0.130243,0.95,"Clinical rule layer поддерживает дефицит, а ML-вероятность низкая."
2,P000005,B12_deficiency,ml_positive_rule_negative,review,0.999954,0.15,Высокая ML-вероятность при слабой поддержке clinical rule layer.
3,P000005,inflammation_anemia,ml_negative_rule_positive,review,0.000122,0.95,"Clinical rule layer поддерживает дефицит, а ML-вероятность низкая."
4,P000009,B12_deficiency,ml_positive_rule_negative,review,0.999510,0.15,Высокая ML-вероятность при слабой поддержке clinical rule layer.
5,P000013,iron_deficiency,ml_positive_rule_negative,review,0.998939,0.05,Высокая ML-вероятность при слабой поддержке clinical rule layer.
6,P000013,inflammation_anemia,ml_negative_rule_positive,review,0.000070,0.95,"Clinical rule layer поддерживает дефицит, а ML-вероятность низкая."
7,P000015,inflammation_anemia,ml_negative_rule_positive,review,0.000041,0.95,"Clinical rule layer поддерживает дефицит, а ML-вероятность низкая."
8,P000016,iron_deficiency,ml_positive_rule_negative,review,0.752516,0.10,Высокая ML-вероятность при слабой поддержке clinical rule layer.
9,P000017,B12_deficiency,ml_positive_rule_negative,review,0.999953,0.15,Высокая ML-вероятность при слабой поддержке clinical rule layer.



### Как интерпретировать conflict

Мы используем продуктовые границы:

- `ensemble probability >= 0.75`, но `rule_score <= 0.25`;
- либо `ensemble probability <= 0.25`, но `rule_score >= 0.75`.

Эти `0.75/0.25` **не медицинские диагностические пороги**.
Это пороги только для флага «ML и clinical evidence сильно расходятся».


## 9. Частота рекомендуемых дополнительных анализов

In [9]:

test_rows = []

for payload in payloads:
    for item in payload["recommended_next_tests"]:
        test_rows.append({
            "patient_id": payload["patient_id"],
            **item,
        })

recommended_tests_df = pd.DataFrame(test_rows)

if len(recommended_tests_df) == 0:
    print("Нет рекомендаций по дополнительным анализам.")
else:
    display(
        recommended_tests_df["test"]
        .value_counts()
        .rename_axis("test")
        .reset_index(name="n")
    )


,test,n
0,Ret_He,210
1,homocysteine,159
2,active_B12,101
3,MMA,90
4,TSAT,45
5,ferritin,25
6,vitamin_B12,22
7,ceruloplasmin,7
8,folate,5
9,copper,1



## 10. Проверяем explanation layer на нескольких типичных случаях

Выбираем:

- самый уверенный прогноз;
- самый неуверенный прогноз;
- пациента с конфликтом, если такой есть.

Это ручная sanity-check проверка перед API.


In [10]:

top_prob = work.apply(
    lambda r: max(patient_probabilities(r).values()),
    axis=1,
)

most_confident_idx = top_prob.idxmax()
least_confident_idx = top_prob.idxmin()

examples = {
    "most_confident": build_explanation_payload(work.loc[most_confident_idx]),
    "least_confident": build_explanation_payload(work.loc[least_confident_idx]),
}

if len(conflict_df) > 0:
    conflict_patient = conflict_df.iloc[0]["patient_id"]
    conflict_row = work.loc[work["patient_id"] == conflict_patient].iloc[0]
    examples["with_conflict"] = build_explanation_payload(conflict_row)

for name, payload in examples.items():
    print("\n" + "=" * 80)
    print(name)
    print(json.dumps(payload, ensure_ascii=False, indent=2))



most_confident
{
  "patient_id": "P000393",
  "prediction": {
    "anemia_class": "copper_deficiency",
    "deficiency_cause": "copper_deficiency",
    "anemia": true
  },
  "confidence": {
    "top_binary_target": "copper_deficiency",
    "top_probability": 1.0,
    "level": "high"
  },
  "deficiencies": {
    "iron_deficiency": {
      "probability": 0.0002,
      "prediction": false,
      "rule_score": 0.05
    },
    "B12_deficiency": {
      "probability": 0.0042,
      "prediction": false,
      "rule_score": 0.5
    },
    "folate_deficiency": {
      "probability": 0.0055,
      "prediction": false,
      "rule_score": 0.1
    },
    "B6_deficiency": {
      "probability": 0.0014,
      "prediction": false,
      "rule_score": 0.5
    },
    "copper_deficiency": {
      "probability": 1.0,
      "prediction": true,
      "rule_score": 0.5
    },
    "inflammation_anemia": {
      "probability": 0.0002,
      "prediction": false,
      "rule_score": 0.95
    }
  },
  "evidence

## 11. Сохраняем explanation artifacts

In [11]:

def resolve_explanation_artifact_root():
    candidates = [
        Path("../artifacts/explanations"),
        Path("artifacts/explanations"),
        Path("../../artifacts/explanations"),
    ]
    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate.resolve()
        except Exception:
            pass
    raise RuntimeError("Не удалось создать artifacts/explanations")

EXPLANATION_DIR = resolve_explanation_artifact_root()

# JSONL удобно использовать в API/ETL.
jsonl_path = EXPLANATION_DIR / "explanation_payloads.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for payload in payloads:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

if len(conflict_df) > 0:
    conflict_df.to_csv(
        EXPLANATION_DIR / "conflicts.csv",
        index=False,
    )
else:
    pd.DataFrame(
        columns=[
            "patient_id",
            "target",
            "type",
            "severity",
            "ensemble_probability",
            "rule_score",
            "message",
        ]
    ).to_csv(
        EXPLANATION_DIR / "conflicts.csv",
        index=False,
    )

if len(recommended_tests_df) > 0:
    recommended_tests_df.to_csv(
        EXPLANATION_DIR / "recommended_next_tests.csv",
        index=False,
    )

print("Saved to:", EXPLANATION_DIR)
for p in sorted(EXPLANATION_DIR.iterdir()):
    print(" -", p.name)


Saved to: /home/sinopah/meditron/artifacts/explanations
 - conflicts.csv
 - explanation_payloads.jsonl
 - recommended_next_tests.csv



# 12. Что проверяем после запуска

Перед внешним датасетом нужно посмотреть три вещи:

1. **Количество и типы conflicts.**  
   Если их очень много — conflict thresholds или сами правила требуют пересмотра.

2. **Примеры payload вручную.**  
   Особенно `most_confident`, `least_confident` и `with_conflict`.

3. **recommended_next_tests.**  
   Проверяем с врачом, что система не предлагает лишние/нелогичные обследования.

После этой проверки:

```text
06_external_validation.ipynb
```

где впервые проверяем transfer/generalization на данных, которые не участвовали
ни в feature selection, ни в обучении, ни в calibration, ни в выборе thresholds.
